# 03 — Entrenamiento y Evaluación: YOLOv8n vs YOLOv8s
**Proyecto:** Placas-Deteccion-Robo
**Puntos del PDF:** 5) Entrenamiento y Evaluación de Modelos, 6) Selección y Exportación del Modelo Final

Continúa desde `02_preprocessing.ipynb` (mismo Colab, ya existe `data/processed/data.yaml`).

Comparación (Opción A del plan, la más simple): **YOLOv8n** (nano) vs **YOLOv8s** (small), fine-tuned pocas épocas (20–30) sobre el dataset chico de placas.


In [5]:
import os

# Si corre en Colab: monta Google Drive y se para en la carpeta ml-service
# ahi adentro, para que 01/02/03 compartan data/ y model/ entre runtimes
# distintos. Si corre local (VS Code), solo ajusta el cwd a ml-service/.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    # Ajusta esta ruta si pusiste la carpeta en otro lugar de tu Drive
    DRIVE_PATH = '/content/drive/MyDrive/Placas-Deteccion-Robo/ml-service'
    os.chdir(DRIVE_PATH)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print('Working directory:', os.getcwd())


Working directory: c:\KAPA-2025\VIco\Placas-Deteccion-Robo\ml-service


### 3.1 Instalación

In [6]:
!pip install -q ultralytics


In [7]:
import torch

print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('ADVERTENCIA: va a entrenar en CPU, va a ser mucho mas lento.')


CUDA disponible: False
ADVERTENCIA: va a entrenar en CPU, va a ser mucho mas lento.


### 3.2 Entrenamiento — YOLOv8n

In [ ]:
from ultralytics import YOLO

model_n = YOLO('yolov8n.pt')  # preentrenado en COCO, se hace fine-tuning

results_n = model_n.train(
    data='data/processed/data.yaml',
    epochs=30,            # antes 5 (muy pocas para converger, ver hallazgo en revisión)
    imgsz=640,
    batch=8,               # antes 16: en GPU local de 4GB VRAM (ej. GTX 1650) evita
                           # quedarse sin memoria a imgsz=640. En Colab (GPU con más
                           # VRAM) se puede volver a subir a 16 si querés acelerar.
    name='placas_yolov8n',
    patience=10,           # early stopping: corta si no mejora en 10 épocas seguidas
    # --- augmentation extra: el dataset son fotos "de catálogo" (autos bien
    # encuadrados, buena luz); en producción la imagen viene de una cámara de
    # calle/celular con ángulos y distancias que el dataset no cubre. Estos
    # parámetros simulan esas condiciones para generalizar mejor: ---
    degrees=10.0,          # antes 0.0: tolera placas levemente rotadas
    perspective=0.0005,    # antes 0.0: simula ángulo de cámara no frontal (rango recomendado 0-0.001)
    mixup=0.15,            # antes 0.0: regularización extra (dataset chico, ~300 imgs train)
)


### 3.3 Entrenamiento — YOLOv8s

In [ ]:
model_s = YOLO('yolov8s.pt')

results_s = model_s.train(
    data='data/processed/data.yaml',
    epochs=30,             # antes 5 (mismo ajuste que YOLOv8n, ver celda anterior)
    imgsz=640,
    batch=8,               # antes 16: YOLOv8s pesa más que el nano, con 4GB de VRAM
                           # local batch=16 probablemente tira CUDA out of memory.
    name='placas_yolov8s',
    patience=10,
    degrees=10.0,
    perspective=0.0005,
    mixup=0.15,
)


### 3.4 Evaluación sobre el set de validación (mAP, precisión, recall)

In [10]:
metrics_n = model_n.val(data='data/processed/data.yaml', split='val')
metrics_s = model_s.val(data='data/processed/data.yaml', split='val')

import pandas as pd

comparacion = pd.DataFrame({
    'modelo': ['YOLOv8n', 'YOLOv8s'],
    'mAP50': [metrics_n.box.map50, metrics_s.box.map50],
    'mAP50-95': [metrics_n.box.map, metrics_s.box.map],
    'precision': [metrics_n.box.mp, metrics_s.box.mp],
    'recall': [metrics_n.box.mr, metrics_s.box.mr],
})
comparacion


Ultralytics 8.3.0  Python-3.10.5 torch-2.13.0+cpu CPU (AMD Ryzen 5 5600GT with Radeon Graphics)
Model summary (fused): 186 layers, 2,684,563 parameters, 0 gradients, 6.8 GFLOPs


val: Scanning C:\KAPA-2025\VIco\Placas-Deteccion-Robo\ml-service\data\processed\labels\val.cache... 86 images, 0 backgrounds, 0 corrupt: 100%|██████████| 86/86 [00:00<?, ?it/s]
c:\KAPA-2025\VIco\Placas-Deteccion-Robo\ml-service\.venv\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:06<00:00,  1.00s/it]


                   all         86         93    0.00341      0.946      0.563      0.292
Speed: 1.2ms preprocess, 53.5ms inference, 0.0ms loss, 2.4ms postprocess per image
Results saved to runs\detect\placas_yolov8n22
Ultralytics 8.3.0  Python-3.10.5 torch-2.13.0+cpu CPU (AMD Ryzen 5 5600GT with Radeon Graphics)
Model summary (fused): 186 layers, 9,828,051 parameters, 0 gradients, 23.3 GFLOPs


val: Scanning C:\KAPA-2025\VIco\Placas-Deteccion-Robo\ml-service\data\processed\labels\val.cache... 86 images, 0 backgrounds, 0 corrupt: 100%|██████████| 86/86 [00:00<?, ?it/s]
c:\KAPA-2025\VIco\Placas-Deteccion-Robo\ml-service\.venv\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:11<00:00,  1.84s/it]


                   all         86         93      0.482     0.0753      0.495      0.278
Speed: 1.2ms preprocess, 112.0ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to runs\detect\placas_yolov8s2


,modelo,mAP50,mAP50-95,precision,recall
0,YOLOv8n,0.563347,0.292285,0.003411,0.946237
1,YOLOv8s,0.495118,0.277980,0.482322,0.075269


### 3.5 Comparación de velocidad de inferencia (relevante para elegir el modelo final)

In [11]:
import time
import glob
import cv2

test_images = glob.glob('data/processed/images/test/*')[:20]

def bench(model, images):
    # warm-up
    _ = model.predict(images[0], verbose=False)
    t0 = time.time()
    for img in images:
        _ = model.predict(img, verbose=False)
    return (time.time() - t0) / len(images)

t_n = bench(model_n, test_images)
t_s = bench(model_s, test_images)

print(f'YOLOv8n: {t_n*1000:.1f} ms/imagen')
print(f'YOLOv8s: {t_s*1000:.1f} ms/imagen')


YOLOv8n: 68.7 ms/imagen
YOLOv8s: 122.4 ms/imagen


### 3.6 Selección del modelo final (completar con los resultados obtenidos)

Criterio de selección: mejor balance entre **mAP@0.5** y **velocidad de inferencia**, priorizando velocidad porque el servicio (`FastAPI /detect`) responde a pedidos síncronos desde el frontend — no hay tiempo para modelos pesados.

- Si la diferencia de mAP@0.5 entre YOLOv8n y YOLOv8s es menor a ~2–3 puntos → **usar YOLOv8n** (más rápido, más liviano para desplegar).
- Si YOLOv8s mejora notablemente el mAP (>5 puntos) y la latencia extra es aceptable (< ~50ms más) → usar YOLOv8s.

`<Completar acá con los números reales de la corrida: mAP50 de cada uno, tiempos de inferencia, y la decisión final en 1 párrafo para el informe técnico>`


### 3.7 Exportación del modelo elegido a ONNX (para el punto 6 y la integración FastAPI del punto 8)

In [ ]:
# Ajustar según cuál haya ganado en 3.6 (acá se deja YOLOv8n como ejemplo por default)
best_model = model_n  # o model_s, según el resultado de la comparación

export_path = best_model.export(format='onnx')
print('Modelo exportado en:', export_path)

# También guardamos el .pt entrenado (útil si se prefiere correr con ultralytics directo en FastAPI en vez de ONNX)
import shutil
shutil.copy('runs/detect/placas_yolov8n2/weights/best.pt', 'model/yolov8n_placas.pt')
shutil.copy('runs/detect/placas_yolov8n2/weights/best.onnx', 'model/yolov8n_placas.onnx')


Ultralytics 8.3.0  Python-3.10.5 torch-2.13.0+cpu CPU (AMD Ryzen 5 5600GT with Radeon Graphics)

PyTorch: starting from 'runs\detect\placas_yolov8n2\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.4 MB)

ONNX: starting export with onnx 1.22.0 opset 10...


W0727 08:55:08.689114 23604 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 10 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 10).
Failed to convert the model to the target version 10 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "c:\KAPA-2025\VIco\Placas-Deteccion-Robo\ml-service\.venv\lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted

ONNX: slimming with onnxslim 0.1.34...
ONNX: export success  5.1s, saved as 'runs\detect\placas_yolov8n2\weights\best.onnx' (10.4 MB)

Export complete (5.2s)
Results saved to C:\KAPA-2025\VIco\Placas-Deteccion-Robo\ml-service\runs\detect\placas_yolov8n2\weights
Predict:         yolo predict task=detect model=runs\detect\placas_yolov8n2\weights\best.onnx imgsz=640  
Validate:        yolo val task=detect model=runs\detect\placas_yolov8n2\weights\best.onnx imgsz=640 data=data/processed/data.yaml  
Visualize:       https://netron.app
Modelo exportado en: runs\detect\placas_yolov8n2\weights\best.onnx


FileNotFoundError: [Errno 2] No such file or directory: 'runs/detect/placas_yolov8n/weights/best.pt'

### 3.8 Descargar los pesos para subirlos a `ml-service/model/` del proyecto

En Colab:
```python
from google.colab import files
files.download('model/yolov8n_placas.pt')
files.download('runs/detect/placas_yolov8n/weights/best.onnx')
```

Copiar ambos archivos a `ml-service/model/` en el repo local antes de levantar FastAPI.
